# Streaming Gradio Q&A chatbot

General-knowledge Q&A with a model dropdown. Tokens stream into Markdown as they arrive.

Needs `OPENAI_API_KEY` in `.env`.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

load_dotenv(override=True)
openai = OpenAI()

In [ ]:
def message_gpt(messages, model):
    stream = openai.chat.completions.create(
        model=model,
        messages=messages,
        stream=True,
    )
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


def qa_chatbot(question, model):
    system_message = "You are a helpful assistant that can answer General Knowledge questions"
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": question},
    ]
    yield from message_gpt(messages, model)

In [ ]:
question_input = gr.Textbox(label="Your question:", info="Ask anything", lines=7)
answer_output = gr.Markdown(label="Answer:")
dropdown = gr.Dropdown(
    ["gpt-4o-mini", "gpt-4.1-mini"],
    label="Select model",
    value="gpt-4o-mini",
)

ui = gr.Interface(
    fn=qa_chatbot,
    inputs=[question_input, dropdown],
    outputs=answer_output,
    flagging_mode="never",
)
ui.launch(inbrowser=True)